In [2]:
import re 
import sys 
import csv
import json
import os
import numpy as np 
import pandas as pd 
from glob import glob 
from typing import List, Any

def read_human_results(folder='/home/work/yuna/HPA/results/check/pilot/human-raw'): 
    dfs=[]
    for f in glob(f'{folder}/*.xlsx'): 
        print(f) 
        df = pd.read_excel(f)
        # preprocess 

        if '4' not in f : 
            df = df.melt(id_vars=df.columns[:3]).rename(columns={'variable': 'question', 'value': 'answer', "이름 또는 이니셜 Name or Initials ": "subj"})
            df['question'] = df['question'].str.replace('scene', 'photo')
            # df['question'] = df['question'].str.split('\n').str[0]
            df[['question_en', 'question (Korean)']] = df['question'].str.split('\n', expand=True)[[0,1]] 
            df['answer'] = df['answer'].replace({'Yes 네': 'yes', "No 아니오": "no"})
            dfs.append(df) 
        else: 
            df = df.melt(id_vars=['Timestamp', "Name or Initials 이름 또는 이니셜 ", 'Score']).rename(columns={'variable': 'question', 'value': 'answer', "Name or Initials 이름 또는 이니셜 ": "subj"})
            hmmm=df 
    df = pd.concat(dfs) 

    return df, hmmm
    
vqa, hmmm = read_human_results()
vqa['subj'] = vqa['subj'].fillna(vqa['Timestamp']) 
hmmm['subj'] = hmmm['subj'].fillna(hmmm['Timestamp']) 
# df = hm.dropna(subset=['qid','answer'], axis=0) 

/home/work/yuna/HPA/results/check/pilot/human-raw/VQA-SPUB Pilot Experiment (Responses).xlsx
/home/work/yuna/HPA/results/check/pilot/human-raw/VQA-SPUB Pilot Experiment Pt.2 (Responses).xlsx
/home/work/yuna/HPA/results/check/pilot/human-raw/VQA-SPUB Pilot Experiment Pt.3 (Responses).xlsx
/home/work/yuna/HPA/results/check/pilot/human-raw/VQA-SPUB Pilot Experiment Pt.4 (Responses).xlsx


In [3]:
# get all the mmstar resutls 
df = hmmm
df['hqid'] = df['question'].factorize()[0] 
selected = pd.read_csv('/home/work/yuna/HPA/eval/mmstar_questions_blind_descending.csv').rename(columns={'Unnamed: 0': 'qid'})
selected.index.name = 'hqid'
selected = selected.reset_index()
print(len(df))
df = pd.merge(selected, df, on =['hqid'], how='right', suffixes=('', '_hm'))
df['correct'] = df['answer_hm'] == df['answer']
print(len(df),len(df.qid.unique()))
df.to_csv('/home/work/yuna/HPA/results/humans/mmstar_pilot.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/home/work/yuna/HPA/eval/mmstar_questions_blind_descending.csv'

In [9]:
# hm = pd.merge(hm, adf, on=['question'], how='left') 
s1 = pd.read_csv('/home/work/yuna/HPA/eda/questions/s1.csv')
hm = pd.merge(vqa, s1, on=['question_en'], how='left', suffixes=('_hm', ''))  
hm = hm.dropna(axis=0, subset=['qid', "answer_hm"])
len(hm)

4381

In [12]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY) # os.getenv("OPENAI_API_KEY"))

def translate_answer(client, question, answer): 
    prompt = f"""
            You are a precise translation and data-formatting assistant.

            Task:
            Given a question in English, return only a simple translation of the korean ANSWER into English. 
            
            Input:
            QUESTION: {question}
            KOREAN_ANSWER: {answer} 
            """

    response = client.chat.completions.create(
        model="gpt-5-nano",   # or your preferred model
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )

    output = response.choices[0].message.content.strip()
    return output

In [15]:
processed_output

'no'

In [ ]:
sys.path.append('/home/work/yuna/HPA/eval')
from korean_translations import korean_translations 
from processor import PostProcessor 
# from scoring import answer_similarity, vqa_accuracy
# from preprocess import contains_korean, get_answer

annotations_path = "/home/work/yuna/VLMEval/data/v2_mscoco_val2014_annotations.json"
with open(os.path.join(annotations_path), 'r') as f: 
    annotations = json.load(f)['annotations'] 

def get_answer(question_id):
    df = pd.DataFrame(annotations)
    df['question_id'] = df['question_id'].astype(int) 
    target_row = df[df['question_id'] == question_id]
    return target_row.iloc[0]['answers']

def contains_korean(text):
    pattern = re.compile(r'[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7A3]')
    return bool(pattern.search(str(text)))

postprocessor = PostProcessor()
for i, row in hm.iterrows():  
    qid = row['qid']
    output = row.get("answer_hm", "") 
    processed_output = postprocessor.postprocess_answer(str(output)) 
    ans = get_answer(int(qid)) 
    
    if contains_korean(processed_output): 
        if not processed_output in korean_translations.keys(): 
            trans = translate_answer(client, row['question_en'], processed_output) 
            print(f'translated {processed_output}: {trans}') 
            korean_translations[processed_output] = trans 
        
        processed_output = korean_translations[processed_output] 
    
    hm.loc[i, 'processed_output'] = processed_output
    ans = [str(a['answer']) for a in ans ]
    acc = vqa_accuracy(ans, processed_output) 
    score = round(answer_similarity(ans, processed_output), 3)
    hm.loc[i, 'acc'] = acc 
    hm.loc[i, 'score'] = score 
    hm.at[i, 'answers'] = ans
hm.to_csv('./vqa_pilot.csv')

In [29]:
with open("/home/work/yuna/HPA/eval/korean_translations.py", "w", encoding="utf-8") as f:
    f.write("korean_translations = ")
    f.write(json.dumps(korean_translations, indent=4, ensure_ascii=False))

In [26]:
len(korean_translations)

432

In [37]:
answer_similarity(['hi', 'hi'], 'hi')

1.0000001192092896

In [ ]:

from analysis.pilot.analysis import vqa_score, str_to_list 
from analysis.utils.question_type_mapper import question_type 